# Airfoil Aerodynamic Coefficient Prediction

Predicting lift (Cl) and drag (Cd) coefficients from airfoil shape parameters (CST coefficients),
using a public OpenFOAM CFD dataset of 2,946 airfoils at Re = 1e5.

Run cells top to bottom — later cells depend on variables defined earlier (`df`, `model`, etc.).


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline


## Stage 1: Load and explore

Put `airfoil_data.csv` in the same folder as this notebook, or adjust the path below.


In [ ]:
df = pd.read_csv("airfoil_data.csv")
df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
df.head()


In [ ]:
print("Dtypes:\n", df.dtypes)
print("\nNulls:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
n_airfoils = df['Filename'].nunique()
print("Unique airfoils:", n_airfoils)

rows_per_airfoil = df.groupby('Filename').size()
print("\nRows per airfoil - describe:")
print(rows_per_airfoil.describe())

print("\nAoA value counts:")
print(df['AoA'].value_counts().sort_index())


In [ ]:
df['L_D'] = df['Cl'] / df['Cd']
print("L/D describe:\n", df['L_D'].describe())
print("\nCd describe (should always be > 0):\n", df['Cd'].describe())
print("Any Cd <= 0?", (df['Cd'] <= 0).sum())


### Sanity-check plots

Lift curve and drag polar for a couple of individual airfoils — the classic aero plots.
Worth eyeballing before trusting the dataset.


In [ ]:
sample_airfoils = df['Filename'].unique()[:3]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name in sample_airfoils:
    sub = df[df['Filename'] == name].sort_values('AoA')
    axes[0].plot(sub['AoA'], sub['Cl'], marker='o', label=name)
    axes[1].plot(sub['Cd'], sub['Cl'], marker='o', label=name)

axes[0].set_xlabel('AoA'); axes[0].set_ylabel('Cl'); axes[0].set_title('Lift curve'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Cd'); axes[1].set_ylabel('Cl'); axes[1].set_title('Drag polar'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## Stage 2: Baseline (linear regression)

**Important:** each airfoil appears in ~11-13 rows (one per angle of attack). A random row-level
train/val split would put the same airfoil shape in both sets, so the model gets evaluated on
shapes it's effectively already seen — that's leakage, and it would make validation performance
look better than true generalization to new shapes.

`GroupShuffleSplit` keeps every row from a given airfoil entirely in train OR val, never both.


In [ ]:
feature_cols = ['AoA'] + [f'CST Coeff {i}' for i in range(1, 9)]
target_cols = ['Cl', 'Cd']

X = df[feature_cols].values
y = df[target_cols].values
groups = df['Filename'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train rows: {len(X_train)}, Val rows: {len(X_val)}")
print(f"Train unique airfoils: {len(set(groups[train_idx]))}")
print(f"Val unique airfoils: {len(set(groups[val_idx]))}")
overlap = set(groups[train_idx]) & set(groups[val_idx])
print(f"Airfoil overlap between train/val: {len(overlap)} (must be 0)")


In [ ]:
scaler_X = StandardScaler().fit(X_train)
X_train_s = scaler_X.transform(X_train)
X_val_s = scaler_X.transform(X_val)

linreg = LinearRegression()
linreg.fit(X_train_s, y_train)
pred_val_linear = linreg.predict(X_val_s)

for i, name in enumerate(target_cols):
    rmse = np.sqrt(mean_squared_error(y_val[:, i], pred_val_linear[:, i]))
    r2 = r2_score(y_val[:, i], pred_val_linear[:, i])
    print(f"Linear {name} — RMSE: {rmse:.5f}, R2: {r2:.4f}")


## Stage 3: Neural network (PyTorch)

Cl and Cd are on very different scales (Cl ~ -1 to 2, Cd ~ 0.001 to 0.09). If we train on raw
values, a single MSE loss summed across both outputs would be dominated by Cl's larger scale and
the model would barely learn to predict Cd. Scaling both inputs and targets with `StandardScaler`
fixes that — we invert the target scaling after prediction to get back to real Cl/Cd units.


In [ ]:
torch.manual_seed(42)

scaler_y = StandardScaler().fit(y_train)
y_train_s = scaler_y.transform(y_train)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)

class AeroNet(nn.Module):
    def __init__(self, in_dim=9, hidden=64, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)

model = AeroNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [ ]:
n_epochs = 300
batch_size = 256
n_train = X_train_t.shape[0]

train_losses = []
for epoch in range(n_epochs):
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for i in range(0, n_train, batch_size):
        idxs = perm[i:i+batch_size]
        xb, yb = X_train_t[idxs], y_train_t[idxs]

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idxs)
    epoch_loss /= n_train
    train_losses.append(epoch_loss)
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  train MSE (scaled): {epoch_loss:.5f}")

plt.plot(train_losses)
plt.xlabel("Epoch"); plt.ylabel("Train MSE (scaled)"); plt.title("Training loss")
plt.show()


In [ ]:
model.eval()
with torch.no_grad():
    pred_val_s = model(X_val_t).numpy()
pred_val_nn = scaler_y.inverse_transform(pred_val_s)

for i, name in enumerate(target_cols):
    rmse = np.sqrt(mean_squared_error(y_val[:, i], pred_val_nn[:, i]))
    r2 = r2_score(y_val[:, i], pred_val_nn[:, i])
    print(f"NN {name} — RMSE: {rmse:.5f}, R2: {r2:.4f}")


## Stage 4: Real evaluation — where is it wrong

One R² number hides everything useful. Break the error down by angle of attack and look at the
worst individual predictions, not just the average.


In [ ]:
val_df = df.iloc[val_idx].copy().reset_index(drop=True)
val_df['Cl_true'] = y_val[:, 0]; val_df['Cd_true'] = y_val[:, 1]
val_df['Cl_pred_nn'] = pred_val_nn[:, 0]; val_df['Cd_pred_nn'] = pred_val_nn[:, 1]
val_df['Cl_abs_err'] = (val_df['Cl_pred_nn'] - val_df['Cl_true']).abs()
val_df['Cd_abs_err'] = (val_df['Cd_pred_nn'] - val_df['Cd_true']).abs()

print("Mean |Cl error| by AoA:")
print(val_df.groupby('AoA')['Cl_abs_err'].mean())

print("\nMean |Cd error| by AoA:")
print(val_df.groupby('AoA')['Cd_abs_err'].mean())


In [ ]:
val_df.nlargest(10, 'Cd_abs_err')[['Filename', 'AoA', 'Cd_true', 'Cd_pred_nn', 'Cd_abs_err']]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(val_df['Cl_true'], val_df['Cl_pred_nn'], s=4, alpha=0.3)
lims = [val_df['Cl_true'].min(), val_df['Cl_true'].max()]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_xlabel('True Cl'); axes[0].set_ylabel('Predicted Cl'); axes[0].set_title('Cl: predicted vs true')

axes[1].scatter(val_df['Cd_true'], val_df['Cd_pred_nn'], s=4, alpha=0.3, color='orange')
lims = [val_df['Cd_true'].min(), val_df['Cd_true'].max()]
axes[1].plot(lims, lims, 'r--', lw=1)
axes[1].set_xlabel('True Cd'); axes[1].set_ylabel('Predicted Cd'); axes[1].set_title('Cd: predicted vs true')

err_by_aoa = val_df.groupby('AoA')['Cd_abs_err'].mean()
axes[2].bar(err_by_aoa.index, err_by_aoa.values, color='steelblue')
axes[2].set_xlabel('AoA'); axes[2].set_ylabel('Mean |Cd error|'); axes[2].set_title('Cd error by angle of attack')

plt.tight_layout()
plt.show()


### What to look for

- **Cl plot**: points should hug the diagonal closely — check if they do
- **Cd plot**: does the model compress high-Cd (high-drag) cases toward lower predicted values?
- **Error-by-AoA bars**: does error increase toward higher AoA (closer to stall)? That's physically
  expected — aerodynamics gets more nonlinear near flow separation — not necessarily a model flaw.
